# Unit 5 — n8n Agentic Workflow Automation
## AI-Based Defect Reporting System
### Agentic AI & Automation — Symbiosis International University

**Learning Objectives (CO5):**
- Build Agentic Workflows Using n8n
- Integrate Gmail, Google Sheets, Google Calendar
- Parse structured JSON outputs
- Automate defect notifications end-to-end


In [ ]:
import os, sys, json
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')
print('✅ Environment loaded')
print(f'N8N_WEBHOOK_URL: {os.getenv("N8N_WEBHOOK_URL", "Not configured (will log locally)")}')

## n8n Architecture Overview

In [ ]:
print('''
n8n Workflow Architecture for AI Defect Reporting:

┌─────────────────────────────────────────────────────────┐
│              n8n Workflow: defect-report                │
│                                                         │
│  [Webhook Trigger]  ←── POST /defect-report             │
│         │                                               │
│         ▼                                               │
│  [Parse & Enrich Data] (JavaScript node)                │
│         │                                               │
│    ┌────┴──────────┬──────────────────┐                 │
│    ▼               ▼                  ▼                 │
│  [Google       [Gmail]           [Google Calendar]      │
│   Sheets]    (HTML Email)       (Review Meeting)        │
│  (Log defect)  (Notify QA)      (Next day 10AM)        │
│                    │                                    │
│                    ▼                                    │
│           [Respond to Webhook]                          │
│           {success: true, actions: [...]}               │
└─────────────────────────────────────────────────────────┘

To set up:
  1. Install n8n: npm install -g n8n
  2. Start n8n: n8n start
  3. Import: n8n/defect_reporting_workflow.json
  4. Add Google credentials (OAuth2)
  5. Set N8N_WEBHOOK_URL in .env
''')

## Sending a Defect to n8n (or Local Fallback)

In [ ]:
from app.tools.n8n_tool import send_defect_to_n8n

# Send a test defect report
print('📡 Sending defect to n8n automation...')
result = send_defect_to_n8n(
    session_id='nb5-demo-001',
    title='Database connection pool exhausted',
    description='Under high load, the database connection pool is exhausted causing 503 errors.',
    component='Database',
    severity='Critical',
    reporter='Unit 5 Demo',
    report_markdown='## Critical Defect\n**Severity:** Critical\n**Component:** Database'
)
print(result)

## Structured Output Parsing (JSON)

In [ ]:
# Demonstrate structured JSON output that n8n receives
import json
from datetime import datetime

structured_payload = {
    'session_id': 'nb5-demo-001',
    'title': 'Database connection pool exhausted',
    'description': 'Under high load, the DB pool is exhausted causing 503 errors for all users.',
    'component': 'Database',
    'severity': 'Critical',
    'reporter': 'AI Analysis System',
    'timestamp': datetime.utcnow().isoformat(),
    'source': 'AI-Based Defect Reporting System',
}

print('📊 Structured JSON payload sent to n8n:')
print(json.dumps(structured_payload, indent=2))

# Simulate n8n response parsing
print('\n📥 n8n response (parsed):')
n8n_response = {
    'success': True,
    'actions': ['logged_to_sheets', 'email_sent', 'calendar_created'],
    'session_id': structured_payload['session_id']
}
print(json.dumps(n8n_response, indent=2))

## View Local Log (When n8n Not Running)

In [ ]:
import json
log_file = '../defect_reports_log.json'

try:
    with open(log_file, 'r') as f:
        logs = json.load(f)
    print(f'📋 Local defect log: {len(logs)} entries\n')
    for log in logs[-3:]:  # Show last 3
        print(f'  [{log.get("timestamp", "N/A")[:19]}]')
        print(f'   Title: {log.get("title", "N/A")}')
        print(f'   Severity: {log.get("severity", "N/A")} | Component: {log.get("component", "N/A")}')
        print()
except FileNotFoundError:
    print('No local log yet — run the cells above first!')

## Importing the n8n Workflow

In [ ]:
import json

with open('../n8n/defect_reporting_workflow.json', 'r') as f:
    workflow = json.load(f)

print(f'n8n Workflow: {workflow["name"]}')
print(f'Nodes ({len(workflow["nodes"])}):')
for node in workflow['nodes']:
    print(f'  - {node["name"]} ({node["type"].split(".")[-1]})')

print()
print('✅ Import steps:')
print('  1. n8n → Workflows → Import from File')
print('  2. Select n8n/defect_reporting_workflow.json')
print('  3. Configure Google credentials')
print('  4. Activate the workflow')
print('  5. Copy webhook URL → paste in .env as N8N_WEBHOOK_URL')